In [53]:
import sys
sys.path.append("..")

from extract.api import coletar_dados
import pandas as pd

# resultado = buscar_preco(59, 5585, "2011-3")
resultado = coletar_dados()
df = pd.DataFrame(resultado)
df

ImportError: cannot import name 'coletar_dados' from 'extract.api' (c:\Users\marce\OneDrive\Documentos\Área de Trabalho\etl_fipe\pipeline-etl-fipe\notebooks\..\extract\api.py)

In [ ]:
def limpar_valor(valor_str):
    valor = valor_str.replace("R$ ", "")
    valor = valor.replace(".", "")
    valor = valor.replace(",", ".")
    return float(valor)

df["Valor"] = df["Valor"].apply(limpar_valor)
df

,TipoVeiculo,Valor,Marca,Modelo,AnoModelo,Combustivel,CodigoFipe,MesReferencia,SiglaCombustivel
0,1,62099.0,VW - VolksWagen,AMAROK CD2.0 16V/S CD2.0 16V TDI 4x2 Die,2011,Diesel,005329-5,julho de 2026,D


In [ ]:
def limpar_marca(marca_str):
    partes = marca_str.split(" - ")
    return {
        "prefixo": partes[0].strip(),
        "marca":   partes[1].strip()
    }

# expande em duas colunas
marca_expandida = df["Marca"].apply(limpar_marca).apply(pd.Series)
df["prefixo"] = marca_expandida["prefixo"]
df["marca"]   = marca_expandida["marca"]
df = df.drop(columns=["Marca"])
df

,TipoVeiculo,Valor,Modelo,AnoModelo,Combustivel,CodigoFipe,MesReferencia,SiglaCombustivel,prefixo,marca
0,1,62099.0,AMAROK CD2.0 16V/S CD2.0 16V TDI 4x2 Die,2011,Diesel,005329-5,julho de 2026,D,VW,VolksWagen


In [ ]:
def limpar_data(mes_ref_str):
    partes = mes_ref_str.split()
    return {
        "mes_nome": partes[0].strip(),
        "ano_referencia": int(partes[2].strip())
    }


data_limpar = df["MesReferencia"].apply(limpar_data).apply(pd.Series)
df["mes_nome"] = data_limpar["mes_nome"]
df["ano_referencia"] = data_limpar["ano_referencia"]
df = df.drop(columns=["MesReferencia"])
df

,TipoVeiculo,Valor,Modelo,AnoModelo,Combustivel,CodigoFipe,SiglaCombustivel,prefixo,marca,mes_nome,ano_referencia
0,1,62099.0,AMAROK CD2.0 16V/S CD2.0 16V TDI 4x2 Die,2011,Diesel,005329-5,D,VW,VolksWagen,julho,2026


In [ ]:
def extrair_modelo(modelo_str):
    partes = modelo_str.split()
    return partes[0]

df["modelo"] = df["Modelo"].apply(extrair_modelo)
df

,TipoVeiculo,Valor,Modelo,AnoModelo,Combustivel,CodigoFipe,SiglaCombustivel,prefixo,marca,mes_nome,ano_referencia,modelo
0,1,62099.0,AMAROK CD2.0 16V/S CD2.0 16V TDI 4x2 Die,2011,Diesel,005329-5,D,VW,VolksWagen,julho,2026,AMAROK


In [ ]:
import re 
def extrair_tracao(tracao_str):
    resultado = re.search(r"4x\d", tracao_str)
    if resultado:
        return resultado.group()
    return None

df["Tracao"] = df["Modelo"].apply(extrair_tracao)
df

,TipoVeiculo,Valor,Modelo,AnoModelo,Combustivel,CodigoFipe,SiglaCombustivel,prefixo,marca,mes_nome,ano_referencia,modelo,Tracao
0,1,62099.0,AMAROK CD2.0 16V/S CD2.0 16V TDI 4x2 Die,2011,Diesel,005329-5,D,VW,VolksWagen,julho,2026,AMAROK,4x2


In [ ]:
def extrair_cambio(cambio_str):
    if re.search(r"Aut", cambio_str):
        return "Automatico"
    return "Manual"

df["cambio"] = df["Modelo"].apply(extrair_cambio)
df

,TipoVeiculo,Valor,Modelo,AnoModelo,Combustivel,CodigoFipe,SiglaCombustivel,prefixo,marca,mes_nome,ano_referencia,modelo,Tracao,cambio
0,1,62099.0,AMAROK CD2.0 16V/S CD2.0 16V TDI 4x2 Die,2011,Diesel,005329-5,D,VW,VolksWagen,julho,2026,AMAROK,4x2,Manual


In [ ]:
def extrair_cilidrada(cilindrada_str):
    resultado = re.search(r"\d+\.\d+",cilindrada_str)
    if resultado:
        return resultado.group()
    return None

df["cilindrada"] = df["Modelo"].apply(extrair_cilidrada)
df = df.drop(columns=["Modelo"])
df

,TipoVeiculo,Valor,AnoModelo,Combustivel,CodigoFipe,SiglaCombustivel,prefixo,marca,mes_nome,ano_referencia,modelo,Tracao,cambio,cilindrada
0,1,62099.0,2011,Diesel,005329-5,D,VW,VolksWagen,julho,2026,AMAROK,4x2,Manual,2.0


In [ ]:
def mapear_tipo(tipo):
    mapa = {1: "carro", 2:"moto",3:"caminhão"}
    return mapa.get(tipo,"desconhecido")

df["TipoVeiculo"] = df["TipoVeiculo"].apply(mapear_tipo)
df = df.drop(columns=["SiglaCombustivel"])
df

,TipoVeiculo,Valor,AnoModelo,Combustivel,CodigoFipe,prefixo,marca,mes_nome,ano_referencia,modelo,Tracao,cambio,cilindrada
0,carro,62099.0,2011,Diesel,005329-5,VW,VolksWagen,julho,2026,AMAROK,4x2,Manual,2.0


In [ ]:
def renomear_colunas(df):
    df = df.rename(columns={
        "TipoVeiculo": "tipo_veiculo",
        "AnoModelo":   "ano_modelo",
        "CodigoFipe":  "codigo_fipe",
        "Tracao":      "tracao",
        "Valor":       "valor",
        "Combustivel": "combustivel"
    })
    df = df[[
        "codigo_fipe", "tipo_veiculo", "modelo", "marca",
        "valor", "prefixo", "ano_modelo", "tracao",
        "cambio", "cilindrada", "combustivel", "ano_referencia"
    ]]
    return df

df = renomear_colunas(df)
df

,codigo_fipe,tipo_veiculo,modelo,marca,valor,prefixo,ano_modelo,tracao,cambio,cilindrada,combustivel,ano_referencia
0,005329-5,carro,AMAROK,VolksWagen,62099.0,VW,2011,4x2,Manual,2.0,Diesel,2026


In [ ]:
registros = []